# Ensemble Generation of Meteorological Forcing Data for SVS

This script generates an ensemble of perturbed meteorological forcing datasets for use in Soil Vegetation Snow (SVS) modeling. Perturbations are applied to key variables based on the methodology outlined in Charrois et al. (2016), incorporating biases and standard deviations derived from comparisons with on-site weather station data.

## Methodology

1. **Data Loading and Preprocessing:**
   - Loads on-site weather station data (air temperature, dew temperature, wind speed, etc.) from a CSV file.
   - Loads original forcing data for the SVS model from another CSV file.
   - Handles missing values and inconsistencies between datasets.

2. **Bias and Standard Deviation Calculation:**
   - Calculates the bias and standard deviation for selected meteorological variables by comparing the on-site data with the original forcing data.
   - Determines whether each variable should be perturbed additively or multiplicatively based on predefined rules.

3. **Autoregressive Modeling of Residuals:**
   - Fits a first-order autoregressive (AR1) model to the residuals (differences) between on-site and forcing data for each variable.
   - Stores the estimated AR1 parameter (phi) for later use in perturbation.

4. **Ensemble Generation (Parallel Processing):**
   - Uses parallel processing (`concurrent.futures.ProcessPoolExecutor`) to generate ensemble members efficiently.
   - For each ensemble member:
     - Perturbs the selected meteorological variables in the original forcing data using the calculated biases, standard deviations, and AR1 parameters.
     - Applies additional perturbations to precipitation (multiplicative) and longwave radiation (additive).
     - Optionally applies corrections to ensure physically plausible values.
     - Saves the perturbed dataset as a compressed Feather file.

## File Paths

* **Input Data:**
    - On-site weather station data: `../Data/OnSiteAveragedMetData.csv`
    - Original forcing data: `../Data/OriginalForcingData.csv`
* **Output Directory:**
    - Ensemble datasets: `./Results/Met_Ens/`

## Parameters

- `N_ENSEMBLE`: Number of ensemble members to generate (default: 30).
- `NCPUS`: Number of CPU cores to use for parallel processing (adjust based on your system).
- `LOCAL_TZ`: Timezone of the on-site weather station data (default: "America/Montreal").

## Key Functions (in `helpers` module)

- `calc_bias_std`: Calculates bias and standard deviation for selected variables.
- `get_ar1_param`: Fits an AR1 model to residuals.
- `perturb_and_save`: Perturbs a forcing dataset and saves it to a file.

## References

- Charrois, L., et al. (2016). On the assimilation of optical reflectances and snow depth observations into a detailed snowpack model. The Cryosphere, 10(3), pp.1021-1038.

## Author

Alireza Amani (alireza.amani101@gmail.com)

## Date

2024-07-22


In [ ]:
# 0) Notes ---------------------------------------------------------------------
'''
In this notebook, we will generate an ensemble of forcing data for the SVS model
by perturbing the original forcing data.

The perturbation method is based on:
Charrois, L., Cosme, E., Dumont, M., Lafaysse, M., Morin, S., Libois, Q. and Picard, G., 2016.
On the assimilation of optical reflectances and snow depth observations into a detailed snowpack model.
The Cryosphere, 10(3), pp.1021-1038.



Created by: Alireza Amani
Email: alireza.amani101@gmail.com
'''
# ______________________________________________________________________________

# 1) Importing Libraries -------------------------------------------------------
from pathlib import Path
import concurrent.futures
import pandas as pd
import helpers
# ______________________________________________________________________________
# 2) Parameters and Variables --------------------------------------------------

## paths
FIELD_STATIONS = Path("../Data/OnSiteAveragedMetData.csv")
FORCING_DATA = Path("../Data/OriginalForcingData.csv")
SAVE_ENS_DIR = Path("./Results/Met_Ens/")

## number of ensemble members
N_ENSEMBLE = 30

## number of cpus for parallel generation of ensemble
NCPUS = 8

## local time zone
LOCAL_TZ = "America/Montreal"


## assertions
assert FIELD_STATIONS.exists(), f"Field stations file does not exist: {FIELD_STATIONS}"
assert FORCING_DATA.exists(), f"Forcing data file does not exist: {FORCING_DATA}"
assert SAVE_ENS_DIR.exists(), f"Save ensemble directory does not exist: {SAVE_ENS_DIR}"

# ______________________________________________________________________________

# 3) Load and Process Data -----------------------------------------------------------------

## load on-site data
field_stations = pd.read_csv(FIELD_STATIONS, index_col=0, parse_dates=True)

### check tz of index
print("Field stations index tz:", field_stations.index.tz)

### set zero wind speed to nan; too many zeros, suspicious.
field_stations["Wind Speed (m/s)"] = field_stations["Wind Speed (m/s)"].replace(0, pd.NA)


## load forcing data
forcing_data = pd.read_csv(FORCING_DATA, index_col=0, parse_dates=True)

### heck tz of index
print("Forcing data index tz:", forcing_data.index.tz)

## compare columns
helpers.compare_dataframe_columns(field_stations, forcing_data)

# ______________________________________________________________________________

# 4) Generate Ensemble ---------------------------------------------------------

## first step is to cacl bias/sdev of select forcing variables wrt field stations
### select variables and their perturbation type (additive or multiplicative)
selvars = {
    "Air Temperature (degC)": {
        "additive": True,
    },
    "Dew Temperature (degC)": {
        "additive": True,
    },
    "Wind Speed (m/s)": {
        "additive": True,
        "var_min": 0.0
    },
    "Shortwave Radiation (W/m2)": {
        "additive": False,
    },
    "Relative Humidity (%)": {
        "additive": False,
        "var_max": 100
    },
    "Atmospheric Pressure (Pa)": {
        "additive": True,
    },
}
bias_std, var_info = helpers.calc_bias_std(
    list(selvars), field_stations, forcing_data
)

## fit first order autoregressive model to residuals
for var_name in bias_std:
    phi = helpers.get_ar1_param(var_info[var_name]['bias'])
    bias_std[var_name]['phi'] = phi

## column labels for each forcing variable
meteo_cols = {
    "utc_dtime": "datetime_utc",
    "air_temperature": "Air Temperature (degC)",
    "dew_point": "Dew Temperature (degC)",
    "precipitation": "Precipitation (mm)",
    "wind_speed": "Wind Speed (m/s)",
    "atmospheric_pressure": "Atmospheric Pressure (Pa)",
    "shortwave_radiation": "Shortwave Radiation (W/m2)",
    "longwave_radiation": "Longwave Radiation (W/m2)",
    "specific_humidity": "Specific Humidity (kg/kg)",
    "relative_humidity": "Relative Humidity (%)"
}

def main():
    # Define the number of workers based on your machine's capabilities
    num_workers = NCPUS

    with concurrent.futures.ProcessPoolExecutor(max_workers=num_workers) as executor:
        futures = [
            executor.submit(
                helpers.perturb_and_save, imem, forcing_data, selvars, bias_std,
                SAVE_ENS_DIR,
                meteo_cols, prec_mult_fac=0.05, longwave_add_pert=25,
                save_as_feather=True, compress_met_file=False,
                catch_corr=True,
            ) for imem in range(N_ENSEMBLE)]

        # Optionally, get the results if you need to process them
        for future in concurrent.futures.as_completed(futures):
            print(future.result())

# ______________________________________________________________________________

# 5) Run the main function -----------------------------------------------------
if __name__ == "__main__":
    main()
# ______________________________________________________________________________